# OmniVoice Studio — Kaggle Dual-T4 + AI-native Server

Optimized for Kaggle sessions exposing **2× Tesla T4 (~15 GiB each), ~30 GiB RAM, and local SSD**.

```text
cuda:0  → OmniVoice TTS
cuda:1  → Whisper ASR verification
CPU/RAM → preprocessing, API/MCP, Gradio, file I/O
SSD     → /kaggle/working/OmniVoiceStudio + transient caches
```

The unified server exposes Gradio `/ui`, REST `/api/v1`, SSE job progress, and MCP `/mcp` from one process. Google Drive/rclone persistence remains intentionally separate from the generation hot path.


In [ ]:
# Kaggle Internet must be enabled for GitHub/Hugging Face downloads.
import os
from pathlib import Path

WORKSPACE = "/kaggle/working/OmniVoiceStudio"
CACHE_ROOT = "/kaggle/working/.cache"
Path(WORKSPACE).mkdir(parents=True, exist_ok=True)
Path(CACHE_ROOT).mkdir(parents=True, exist_ok=True)

# Keep caches on Kaggle local SSD and reduce CUDA memory fragmentation during long queues.
os.environ["HF_HOME"] = f"{CACHE_ROOT}/huggingface"
os.environ["TORCH_HOME"] = f"{CACHE_ROOT}/torch"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install -q --upgrade "git+https://github.com/binhminhanh1235/OmniVoice.git@master"


In [ ]:
import shutil
import torch

from omnivoice.hardware_quality import (
    HardwareQualitySettingsStore,
    detect_hardware,
)
from omnivoice.runtime_workspace import detect_runtime_workspace, ensure_runtime_workspace

runtime = ensure_runtime_workspace(detect_runtime_workspace())
if runtime.environment != "kaggle":
    raise RuntimeError(f"Expected Kaggle runtime, detected: {runtime.environment}")
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU T4 x2 in Kaggle Notebook settings.")

GPU_COUNT = torch.cuda.device_count()
for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {props.total_memory / 1024**3:.1f} GiB")

TTS_DEVICE = "cuda:0"
ASR_DEVICE = "cuda:1" if GPU_COUNT >= 2 else "cpu"
ASR_MODEL = "openai/whisper-small.en"

hardware = detect_hardware(device_index=0)
HardwareQualitySettingsStore(WORKSPACE).set_default(hardware.recommended_preset)

print("Runtime:", runtime.summary())
print("Hardware:", hardware.summary())
for note in hardware.notes:
    print("-", note)
print("OmniVoice device:", TTS_DEVICE)
print("Whisper ASR device:", ASR_DEVICE)
print("Whisper ASR model:", ASR_MODEL)
print("Default quality preset:", hardware.recommended_preset)
print("Workspace:", WORKSPACE)
usage = shutil.disk_usage("/kaggle/working")
print(f"Local SSD free: {usage.free / 1024**3:.1f} GiB")


## Why this mapping?

OmniVoice stays entirely on `cuda:0` instead of being sharded across both T4s. This avoids inter-GPU transfer overhead and keeps the generation path simple. `cuda:1` becomes a dedicated Whisper accelerator, which is especially useful because Project Studio may verify every generated chunk and request word timestamps for pacing checks.

`openai/whisper-small.en` remains the default for English narration. A stricter verifier can use `openai/whisper-medium.en` on GPU1 without touching the TTS GPU.


## Optional: stable hostname + private access

For ChatGPT / Claude Code / Antigravity, create one remotely-managed Cloudflare Tunnel and map a stable hostname such as `omnivoice.example.com` to `http://localhost:8000`.

Create these Kaggle Secrets when using the stable tunnel:
- `CLOUDFLARE_TUNNEL_TOKEN`
- `OMNIVOICE_API_TOKEN`
- `OMNIVOICE_UI_USERNAME`
- `OMNIVOICE_UI_PASSWORD`


In [ ]:
PUBLIC_URL = "https://omnivoice.example.com"
USE_STABLE_TUNNEL = True

if USE_STABLE_TUNNEL:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    required = {
        "CLOUDFLARE_TUNNEL_TOKEN": secrets.get_secret("CLOUDFLARE_TUNNEL_TOKEN"),
        "OMNIVOICE_API_TOKEN": secrets.get_secret("OMNIVOICE_API_TOKEN"),
        "OMNIVOICE_UI_USERNAME": secrets.get_secret("OMNIVOICE_UI_USERNAME"),
        "OMNIVOICE_UI_PASSWORD": secrets.get_secret("OMNIVOICE_UI_PASSWORD"),
    }
    missing = [name for name, value in required.items() if not value]
    if missing:
        raise RuntimeError("Missing Kaggle Secrets: " + ", ".join(missing))
    for name, value in required.items():
        os.environ[name] = value
    os.environ["OMNIVOICE_API_TOKEN_SCOPES"] = (
        "omnivoice:read,omnivoice:generate,omnivoice:queue,omnivoice:mcp"
    )
    os.environ["OMNIVOICE_PUBLIC_URL"] = PUBLIC_URL
    del required, secrets
    print("Stable private public URL configured:", PUBLIC_URL)


In [ ]:
if USE_STABLE_TUNNEL:
    !wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod 700 /kaggle/working/cloudflared
    !/kaggle/working/cloudflared --version


## Launch OmniVoice Studio

With the stable tunnel enabled, one hostname exposes `/ui`, `/api/v1`, `/mcp`, and `/health`. If disabled, the notebook falls back to the temporary Gradio share URL.


In [ ]:
if USE_STABLE_TUNNEL:
    !omnivoice-studio serve \
      --model k2-fsa/OmniVoice \
      --device {TTS_DEVICE} \
      --workspace {WORKSPACE} \
      --asr-model {ASR_MODEL} \
      --asr-device {ASR_DEVICE} \
      --host 0.0.0.0 \
      --port 8000 \
      --tunnel \
      --cloudflared /kaggle/working/cloudflared \
      --public-url {PUBLIC_URL}
else:
    print("Stable tunnel disabled. Using temporary Gradio share URL.")
    !omnivoice-project-studio \
      --model k2-fsa/OmniVoice \
      --device {TTS_DEVICE} \
      --workspace {WORKSPACE} \
      --asr-model {ASR_MODEL} \
      --asr-device {ASR_DEVICE} \
      --share


## Persistence reminder

The dual-GPU mapping and stable hostname improve execution and connectivity, not persistence. Projects, voices, `jobs.json`, `section-status.json`, and queue state remain under `/kaggle/working/OmniVoiceStudio` and disappear when the Kaggle session is discarded.
